In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [ ]:
import pandas as pd

# reemplaza por el nombre real
df = pd.read_csv("cuidados_plantas.csv")

df.head()

In [ ]:
docs = (
    "Plant: " + df["name"] +
    " (" + df["scientific_name"].astype(str) + "). " +
    "Sunlight: " + df["sunlight"].astype(str) + ". " +
    "Watering: " + df["watering"].astype(str) + ". " +
    "Pruning: " + df["pruning"].astype(str)
).tolist()

docs[:3]

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(docs, show_progress_bar=True)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Index size:", index.ntotal)

In [ ]:
def ask_template(question):
    q_emb = embed_model.encode([question])
    D, I = index.search(np.array(q_emb), 1)

    row = df.iloc[I[0][0]]

    return f"""
🌿 {row['name']}
🔬 {row['scientific_name']}

☀️ Luz:
{row['sunlight']}

💧 Riego:
{row['watering']}

✂️ Poda:
{row['pruning']}
"""

In [ ]:
ask_template("How to care of a cockins?")